# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanmustafa119/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### Baseline rule

I will prioritize content for review when it has meaningful Google Search visibility but relatively weak click performance.

The baseline score will use observed March 2026 GSC performance. Pages with more impressions and lower click-through rate will receive a higher review priority.

This is a simple baseline for comparison with future ML models. It is not intended to prove that a page needs a refresh.

### Reason codes

- `HIGH_IMPRESSIONS_LOW_CTR` — the page has substantial search visibility but a relatively low CTR.
- `LOW_CLICKS` — the page receives impressions but very few clicks.
- `REVIEW` — the page is prioritized by the baseline score but does not meet a more specific reason-code condition.

The output is a directional, decision-support ranking for human review.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("Hugging Face authentication successful.")

Hugging Face authentication successful.


In [4]:
import duckdb

con = duckdb.connect()

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

print("DuckDB HTTP support loaded.")

DuckDB HTTP support loaded.


In [5]:
con.execute("DROP SECRET IF EXISTS hf_secret")

con.execute("""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("Hugging Face token configured for DuckDB.")

Hugging Face token configured for DuckDB.


In [6]:
baseline_query = """
SELECT
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
"""

baseline_df = con.sql(baseline_query).df()

print("Rows:", len(baseline_df))
baseline_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 3611061


,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,content_b7e512995f79d5a6,20,0,3.350000
1,content_05597932fe4da067,1,0,0.000000
2,content_7a105f548d9c6916,125,1,4.928000
3,content_905aa32a0230694e,7,0,4.000000
4,content_a3ea9792f793ec72,11,0,2.272727


In [7]:
baseline_df["ctr"] = (
    baseline_df["gsc_clicks"] /
    baseline_df["gsc_impressions"].replace(0, pd.NA)
)

baseline_df[[
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "ctr"
]].head(10)

,content_hash_id,gsc_impressions,gsc_clicks,ctr
0,content_b7e512995f79d5a6,20,0,0.000000
1,content_05597932fe4da067,1,0,0.000000
2,content_7a105f548d9c6916,125,1,0.008000
3,content_905aa32a0230694e,7,0,0.000000
4,content_a3ea9792f793ec72,11,0,0.000000
5,content_36c36abc7650d7af,239,1,0.004184
6,content_a7da352b73b02668,191,0,0.000000
7,content_05434271b257bb68,55,0,0.000000
8,content_d056587ff7faca0c,77,0,0.000000
9,content_bfd1e41c2af250c8,2,0,0.000000


In [8]:
# Create a simple baseline priority score
# Higher impressions + lower CTR = higher review priority

baseline_df["baseline_score"] = (
    baseline_df["gsc_impressions"] * (1 - baseline_df["ctr"])
)

baseline_df[[
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "baseline_score"
]].head(10)

,content_hash_id,gsc_impressions,gsc_clicks,ctr,baseline_score
0,content_b7e512995f79d5a6,20,0,0.000000,20.0
1,content_05597932fe4da067,1,0,0.000000,1.0
2,content_7a105f548d9c6916,125,1,0.008000,124.0
3,content_905aa32a0230694e,7,0,0.000000,7.0
4,content_a3ea9792f793ec72,11,0,0.000000,11.0
5,content_36c36abc7650d7af,239,1,0.004184,238.0
6,content_a7da352b73b02668,191,0,0.000000,191.0
7,content_05434271b257bb68,55,0,0.000000,55.0
8,content_d056587ff7faca0c,77,0,0.000000,77.0
9,content_bfd1e41c2af250c8,2,0,0.000000,2.0


In [9]:
# Start every page with a general review reason
baseline_df["reason_code"] = "REVIEW"

# Pages with high visibility but relatively low CTR
high_impressions = baseline_df["gsc_impressions"].quantile(0.75)
low_ctr = baseline_df["ctr"].median()

baseline_df.loc[
    (baseline_df["gsc_impressions"] >= high_impressions) &
    (baseline_df["ctr"] < low_ctr),
    "reason_code"
] = "HIGH_IMPRESSIONS_LOW_CTR"

# Pages with impressions but very few clicks
baseline_df.loc[
    (baseline_df["gsc_impressions"] > 0) &
    (baseline_df["gsc_clicks"] <= 1),
    "reason_code"
] = "LOW_CLICKS"

baseline_df[[
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "baseline_score",
    "reason_code"
]].head(20)

,content_hash_id,gsc_impressions,gsc_clicks,ctr,baseline_score,reason_code
0,content_b7e512995f79d5a6,20,0,0.000000,20.0,LOW_CLICKS
1,content_05597932fe4da067,1,0,0.000000,1.0,LOW_CLICKS
2,content_7a105f548d9c6916,125,1,0.008000,124.0,LOW_CLICKS
3,content_905aa32a0230694e,7,0,0.000000,7.0,LOW_CLICKS
4,content_a3ea9792f793ec72,11,0,0.000000,11.0,LOW_CLICKS
5,content_36c36abc7650d7af,239,1,0.004184,238.0,LOW_CLICKS
6,content_a7da352b73b02668,191,0,0.000000,191.0,LOW_CLICKS
7,content_05434271b257bb68,55,0,0.000000,55.0,LOW_CLICKS
8,content_d056587ff7faca0c,77,0,0.000000,77.0,LOW_CLICKS
9,content_bfd1e41c2af250c8,2,0,0.000000,2.0,LOW_CLICKS


In [10]:
# Start with a general review reason
baseline_df["reason_code"] = "REVIEW"

# Calculate thresholds
high_impressions = baseline_df["gsc_impressions"].quantile(0.75)
low_ctr = baseline_df["ctr"].median()

# High visibility but low CTR
baseline_df.loc[
    (baseline_df["gsc_impressions"] >= high_impressions) &
    (baseline_df["ctr"] < low_ctr),
    "reason_code"
] = "HIGH_IMPRESSIONS_LOW_CTR"

# Very low clicks only when impressions are meaningful
baseline_df.loc[
    (baseline_df["gsc_impressions"] >= 10) &
    (baseline_df["gsc_clicks"] <= 1) &
    ~(
        (baseline_df["gsc_impressions"] >= high_impressions) &
        (baseline_df["ctr"] < low_ctr)
    ),
    "reason_code"
] = "LOW_CLICKS"

baseline_df[
    [
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "baseline_score",
        "reason_code"
    ]
].head(20)

,content_hash_id,gsc_impressions,gsc_clicks,ctr,baseline_score,reason_code
0,content_b7e512995f79d5a6,20,0,0.000000,20.0,LOW_CLICKS
1,content_05597932fe4da067,1,0,0.000000,1.0,REVIEW
2,content_7a105f548d9c6916,125,1,0.008000,124.0,LOW_CLICKS
3,content_905aa32a0230694e,7,0,0.000000,7.0,REVIEW
4,content_a3ea9792f793ec72,11,0,0.000000,11.0,LOW_CLICKS
5,content_36c36abc7650d7af,239,1,0.004184,238.0,LOW_CLICKS
6,content_a7da352b73b02668,191,0,0.000000,191.0,LOW_CLICKS
7,content_05434271b257bb68,55,0,0.000000,55.0,LOW_CLICKS
8,content_d056587ff7faca0c,77,0,0.000000,77.0,LOW_CLICKS
9,content_bfd1e41c2af250c8,2,0,0.000000,2.0,REVIEW


In [11]:
# Rank pages from highest priority to lowest priority
ranked_df = (
    baseline_df
    .sort_values("baseline_score", ascending=False)
    .reset_index(drop=True)
)

ranked_df["rank"] = ranked_df.index + 1

print("Total ranked pages:", len(ranked_df))

ranked_df.head(20)

Total ranked pages: 3611061


,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,baseline_score,reason_code,rank
0,content_44f34c0a90047651,40084,1,0.083350,0.000025,40083.0,LOW_CLICKS,1
1,content_eadb33b5df496f4a,39305,252,2.197507,0.006411,39053.0,REVIEW,2
2,content_34a70fea29d15f24,39003,2,2.764916,0.000051,39001.0,REVIEW,3
3,content_eadb33b5df496f4a,38436,271,2.195988,0.007051,38165.0,REVIEW,4
4,content_945d6ff91386c817,37368,0,8.613948,0.000000,37368.0,LOW_CLICKS,5
5,content_eadb33b5df496f4a,35404,225,2.188397,0.006355,35179.0,REVIEW,6
6,content_eadb33b5df496f4a,34817,223,2.181348,0.006405,34594.0,REVIEW,7
7,content_eadb33b5df496f4a,34606,235,2.242501,0.006791,34371.0,REVIEW,8
8,content_fec55986a1868d62,33383,0,0.181500,0.000000,33383.0,LOW_CLICKS,9
9,content_eadb33b5df496f4a,33571,215,2.309046,0.006404,33356.0,REVIEW,10


In [12]:
import os

os.makedirs("work/outputs", exist_ok=True)

In [13]:
output_path = "work/outputs/baseline_action_score.csv"

ranked_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(ranked_df))

Saved: work/outputs/baseline_action_score.csv
Rows: 3611061


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

The baseline produces a ranked review queue using observed March 2026 Google Search performance.

The top 20 rows are prioritized using the baseline score. The action for each item is "Review", because the score is a screening rule rather than a validated causal recommendation.

### Review interpretation

- **Action:** Review the content/search performance before taking any action.
- **Reason code:** The reason assigned by the baseline rule.
- **Confidence:** Medium when the row has substantial impressions; lower when the observed click volume is very small.
- **What could make it wrong:** The ranking uses only a small set of GSC signals and does not account for search intent, content quality, business importance, or other information outside the feature set.

The ranking is therefore directional and intended for decision-support rather than automatic content changes.

In [15]:
# Display the Top-20 review queue

top20 = ranked_df[
    [
        "rank",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "baseline_score",
        "reason_code"
    ]
].head(20)

top20

,rank,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,baseline_score,reason_code
0,1,content_44f34c0a90047651,40084,1,0.000025,0.083350,40083.0,LOW_CLICKS
1,2,content_eadb33b5df496f4a,39305,252,0.006411,2.197507,39053.0,REVIEW
2,3,content_34a70fea29d15f24,39003,2,0.000051,2.764916,39001.0,REVIEW
3,4,content_eadb33b5df496f4a,38436,271,0.007051,2.195988,38165.0,REVIEW
4,5,content_945d6ff91386c817,37368,0,0.000000,8.613948,37368.0,LOW_CLICKS
5,6,content_eadb33b5df496f4a,35404,225,0.006355,2.188397,35179.0,REVIEW
6,7,content_eadb33b5df496f4a,34817,223,0.006405,2.181348,34594.0,REVIEW
7,8,content_eadb33b5df496f4a,34606,235,0.006791,2.242501,34371.0,REVIEW
8,9,content_fec55986a1868d62,33383,0,0.000000,0.181500,33383.0,LOW_CLICKS
9,10,content_eadb33b5df496f4a,33571,215,0.006404,2.309046,33356.0,REVIEW


In [16]:
# Add simple confidence notes based on observed impressions

top20_review = top20.copy()

top20_review["action"] = "Review"

top20_review["confidence_note"] = top20_review["gsc_impressions"].apply(
    lambda x: "Higher confidence: substantial observed impressions"
    if x >= 1000
    else "Lower confidence: limited observed impressions"
)

top20_review["what_could_make_it_wrong"] = (
    "Search intent, content quality, or missing context may change the recommendation."
)

top20_review

,rank,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,baseline_score,reason_code,action,confidence_note,what_could_make_it_wrong
0,1,content_44f34c0a90047651,40084,1,0.000025,0.083350,40083.0,LOW_CLICKS,Review,Higher confidence: substantial observed impres...,"Search intent, content quality, or missing con..."
1,2,content_eadb33b5df496f4a,39305,252,0.006411,2.197507,39053.0,REVIEW,Review,Higher confidence: substantial observed impres...,"Search intent, content quality, or missing con..."
2,3,content_34a70fea29d15f24,39003,2,0.000051,2.764916,39001.0,REVIEW,Review,Higher confidence: substantial observed impres...,"Search intent, content quality, or missing con..."
3,4,content_eadb33b5df496f4a,38436,271,0.007051,2.195988,38165.0,REVIEW,Review,Higher confidence: substantial observed impres...,"Search intent, content quality, or missing con..."
4,5,content_945d6ff91386c817,37368,0,0.000000,8.613948,37368.0,LOW_CLICKS,Review,Higher confidence: substantial observed impres...,"Search intent, content quality, or missing con..."
5,6,content_eadb33b5df496f4a,35404,225,0.006355,2.188397,35179.0,REVIEW,Review,Higher confidence: substantial observed impres...,"Search intent, content quality, or missing con..."
6,7,content_eadb33b5df496f4a,34817,223,0.006405,2.181348,34594.0,REVIEW,Review,Higher confidence: substantial observed impres...,"Search intent, content quality, or missing con..."
7,8,content_eadb33b5df496f4a,34606,235,0.006791,2.242501,34371.0,REVIEW,Review,Higher confidence: substantial observed impres...,"Search intent, content quality, or missing con..."
8,9,content_fec55986a1868d62,33383,0,0.000000,0.181500,33383.0,LOW_CLICKS,Review,Higher confidence: substantial observed impres...,"Search intent, content quality, or missing con..."
9,10,content_eadb33b5df496f4a,33571,215,0.006404,2.309046,33356.0,REVIEW,Review,Higher confidence: substantial observed impres...,"Search intent, content quality, or missing con..."


In [17]:
print("Rows in ranked queue:", len(ranked_df))
print("Unique content pages:", ranked_df["content_hash_id"].nunique())

Rows in ranked queue: 3611061
Unique content pages: 176738


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

##  Weak picks + leakage check

The baseline score is a simple decision-support ranking based on observed March 2026 GSC performance. Some high-scoring pages may still be weak recommendations because impressions alone do not establish search intent, content quality, or the reason for low clicks.

Potential weak picks include pages with high impressions but very low clicks where the underlying search intent, SERP context, or content relevance is unknown.

I checked the feature columns used by the baseline and did not include future outcome windows or product flags. The ranking therefore uses only information available in the March 2026 decision period.

This score is directional and should be treated as a review queue rather than proof that a page needs to be changed.

In [18]:
# Leakage check
future_columns = [
    col for col in ranked_df.columns
    if any(term in col.lower() for term in ["future", "april", "may", "june", "label"])
]

print("Possible future/leakage columns:", future_columns)

Possible future/leakage columns: []


## Self-check

Before you submit, confirm each line honestly:

- ✔️ Every section above is filled — markdown thinking AND the code that backs it
- ✔️ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✔️ No client names, URLs, or private queries anywhere
- ✔️ My claims use careful words: observed, measured, directional, decision-support
- ✔️ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.